# Count galaxies per tomographic bin

For each case (`true_nz`, `spec_emulate_nz`) and each `numbin` (1-10) under `equalredshiftwidth/{numbin}_bins/out/`, this notebook:
1. Loads `tomo_bins_wide.hdf5` -> `tomo_bins_wide[:,0]`, the bin assignment for each wide-SOM cell (row index = cell index, 1024 cells).
2. Loads `widedata_wide_assignment.hdf5` -> `cells`, the wide-SOM cell assigned to each wide galaxy.
3. Maps each galaxy's cell to its bin and counts galaxies per tomographic bin.

Bin edges (from `tomobinning_project/Bin edge documentation.txt`, Case 1b = true_nz, Case 2b = spec_emulate_nz) are printed alongside the counts for reference.

In [1]:
import h5py
import numpy as np
import os

In [2]:
outdir = '.'  
binning = 'equalredshiftwidth'
cases = {
    'Case 1b: True z': 'true_nz',
    'Case 2b: Emulated spec-z': 'spec_emulate_nz',
}

# Bin edges copied from "Bin edge documentation.txt" (equal redshift width; identical for both cases)
# Only for printing, not for neff computation
bin_edges_doc = {
    1: [-0.025, 3.075],
    2: [-0.025, 1.525, 3.075],
    3: [-0.025, 1.025, 2.025, 3.075],
    4: [-0.025, 0.775, 1.525, 2.275, 3.075],
    5: [-0.025, 0.575, 1.225, 1.825, 2.475, 3.075],
    6: [-0.025, 0.475, 1.025, 1.525, 2.025, 2.575, 3.075],
    7: [-0.025, 0.425, 0.875, 1.325, 1.725, 2.175, 2.625, 3.075],
    8: [-0.025, 0.375, 0.775, 1.125, 1.525, 1.925, 2.275, 2.675, 3.075],
    9: [-0.025, 0.325, 0.675, 1.025, 1.375, 1.675, 2.025, 2.375, 2.725, 3.075],
    10: [-0.025, 0.275, 0.575, 0.925, 1.225, 1.525, 1.825, 2.125, 2.475, 2.775, 3.075],
}

# Predicted total n_eff for the medium tier (arcmin^-2)
NEFF_TOTAL_MEDIUM_TIER = 41.3

In [3]:
def count_galaxies_per_bin(case_dir, numbin):
    run_dir = os.path.join(outdir, case_dir, binning, f'{numbin}_bins', 'out')

    with h5py.File(os.path.join(run_dir, 'tomo_bins_wide.hdf5'), 'r') as f:
        cell_to_bin = f['tomo_bins_wide'][:, 0].astype(int)  

    with h5py.File(os.path.join(run_dir, 'widedata_wide_assignment.hdf5'), 'r') as f:
        gal_cells = f['cells'][:] 

    gal_bins = cell_to_bin[gal_cells]

    counts = {b: int(np.sum(gal_bins == b)) for b in range(numbin)}
    n_unassigned = int(np.sum(gal_bins == -1))
    total = len(gal_bins)
    return counts, n_unassigned, total


def neff_per_bin(counts, neff_total):
    # Rescale per-bin galaxy counts so they sum to neff_total, assuming n_eff
    # scales with galaxy count (i.e. galaxy weigh the same).
    n_assigned = sum(counts.values())
    return [counts[b] / n_assigned * neff_total for b in range(len(counts))]

In [ ]:
results = {}

for case_label, case_dir in cases.items():
    print(case_label)
    print('=' * len(case_label))
    for numbin in range(1, 11):
        counts, n_unassigned, total = count_galaxies_per_bin(case_dir, numbin)
        neff = neff_per_bin(counts, NEFF_TOTAL_MEDIUM_TIER)
        results[(case_dir, numbin)] = (counts, n_unassigned, total, neff)

        print(f'numbin: {numbin}')
        print('bin_edges: ', bin_edges_doc[numbin])
        print('counts: ', [counts[b] for b in range(numbin)])
        print('neff: ', [round(n, 2) for n in neff])
        print('unassigned galaxies: ', n_unassigned)
        print('total wide galaxies:', total)
        print('=' * 47)
    print()